# Hotel Booking Cancellation Prediction — Random Forest (All Features) + Hyperparameter Tuning

This notebook focuses on a single model, **Random Forest**, using the **full feature set**
(no feature selection), and tunes its hyperparameters using `RandomizedSearchCV`.

Random Forest was chosen because it was the top-performing model in the earlier full comparison
(~0.84 accuracy with default settings). This notebook tries to push that further with tuning.

Leakage columns (`reservation_status`, `reservation_status_date`) are dropped exactly as before —
this is not optional, it's required for the results to be meaningful.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv("/content/hotel_bookings.csv")

print("Dataset Loaded Successfully!")

df.head()


In [ ]:
print("Shape of Dataset :", df.shape)
print("\nMissing Values\n")
print(df.isnull().sum())
print("\nDuplicate Rows :", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates()

print("Duplicate Rows After :", df.duplicated().sum())


In [ ]:
# Fill missing values

if 'children' in df.columns:
    df['children'].fillna(df['children'].median(), inplace=True)

if 'country' in df.columns:
    df['country'].fillna(df['country'].mode()[0], inplace=True)

if 'agent' in df.columns:
    df['agent'].fillna(0, inplace=True)

if 'company' in df.columns:
    df['company'].fillna(0, inplace=True)

print(df.isnull().sum())


In [ ]:
# Feature Engineering

df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df["total_stay"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["total_cost"] = (
    df["adr"] *
    df["total_stay"]
)

df.head()


## Remove Data Leakage

`reservation_status` and `reservation_status_date` are only known **after** the booking outcome
is decided — dropping them is mandatory before any training.


In [ ]:
leak_cols = ["reservation_status", "reservation_status_date"]

leak_cols_present = [c for c in leak_cols if c in df.columns]
print("Dropping leakage columns:", leak_cols_present)

df = df.drop(columns=leak_cols_present)

print("Remaining columns:", df.shape[1])


In [ ]:
# Label Encoding

encoder = LabelEncoder()

cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = encoder.fit_transform(df[col].astype(str))

print("Encoding Completed")


In [ ]:
# Full feature set -- no feature selection

X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

assert not any(c in X.columns for c in leak_cols), "Leakage column found in X!"

print(X.shape)
print(y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Note: Random Forest does not require feature scaling (it's not distance-based),
# but we scale anyway to keep this notebook consistent with the earlier comparison
# notebook. Feel free to skip scaling for RF specifically -- results will be
# effectively identical since trees split on raw thresholds either way.


## Baseline Random Forest (default parameters)

Train once with default settings first, so we have a "before tuning" number to compare against.


In [ ]:
baseline_rf = RandomForestClassifier(random_state=42)

baseline_rf.fit(X_train_scaled, y_train)

baseline_pred = baseline_rf.predict(X_test_scaled)
baseline_prob = baseline_rf.predict_proba(X_test_scaled)[:, 1]

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_prob)

print("Baseline Random Forest (default params)")
print("Test Accuracy :", round(baseline_acc, 4))
print("ROC AUC       :", round(baseline_auc, 4))


## Hyperparameter Tuning with RandomizedSearchCV

We search over a range of Random Forest hyperparameters. `RandomizedSearchCV` is used instead
of `GridSearchCV` because it samples a fixed number of random combinations rather than trying
every single one -- much faster while still covering the search space well.

Only 3-fold CV and a moderate number of iterations are used to keep runtime reasonable
(a few minutes rather than hours). Increase `n_iter` / `cv` later if you want a more thorough
search and have time to spare.


In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
    "class_weight": [None, "balanced"]
}

rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,          # number of random combinations to try
    cv=3,                # 3-fold cross validation
    scoring="accuracy",
    n_jobs=-1,            # use all available CPU cores
    verbose=2,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

print("Best Parameters Found:")
print(random_search.best_params_)

print("\nBest Cross-Validation Accuracy :", round(random_search.best_score_, 4))


In [ ]:
best_rf = random_search.best_estimator_

# Predictions on the held-out test set
y_pred = best_rf.predict(X_test_scaled)
y_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

# Metrics
train_acc = best_rf.score(X_train_scaled, y_train)
test_acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
cv_score = cross_val_score(best_rf, X_train_scaled, y_train, cv=5).mean()

diff = train_acc - test_acc
if diff <= 0.05:
    status = "Good Fit"
elif diff <= 0.10:
    status = "Slight Overfit"
else:
    status = "Overfitting"

print("="*60)
print("Tuned Random Forest -- Final Results")
print("="*60)
print("Train Accuracy :", round(train_acc, 4))
print("Test Accuracy  :", round(test_acc, 4))
print("Precision      :", round(precision, 4))
print("Recall         :", round(recall, 4))
print("F1 Score       :", round(f1, 4))
print("ROC AUC        :", round(roc, 4))
print("CV Score       :", round(cv_score, 4))
print("Status         :", status)


## Before vs After Tuning


In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Test Accuracy", "ROC AUC"],
    "Before Tuning (Default)": [baseline_acc, baseline_auc],
    "After Tuning (Best Params)": [test_acc, roc]
})

comparison["Improvement"] = (
    comparison["After Tuning (Best Params)"] - comparison["Before Tuning (Default)"]
)

comparison


In [ ]:
x = np.arange(len(comparison))
width = 0.35

plt.figure(figsize=(7,5))
plt.bar(x - width/2, comparison["Before Tuning (Default)"], width, label="Before Tuning")
plt.bar(x + width/2, comparison["After Tuning (Best Params)"], width, label="After Tuning")

plt.xticks(x, comparison["Metric"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Random Forest: Before vs After Hyperparameter Tuning")
plt.legend()
plt.show()


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix -- Tuned Random Forest")
plt.show()


In [ ]:
print("="*60)
print("Classification Report -- Tuned Random Forest")
print("="*60)
print(classification_report(y_test, y_pred))


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Tuned Random Forest (AUC = {roc:.4f})")
plt.plot([0,1], [0,1], 'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve -- Tuned Random Forest")
plt.legend()
plt.show()


In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10,6))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Top 15 Feature Importances -- Tuned Random Forest")
plt.xlabel("Importance")
plt.show()

print(importances)


## Test the Tuned Model on 10 Random Records


In [ ]:
sample_data = df.sample(n=10, random_state=42)

X_sample = sample_data.drop("is_canceled", axis=1)
y_actual = sample_data["is_canceled"]

X_sample_scaled = scaler.transform(X_sample)

y_pred_sample = best_rf.predict(X_sample_scaled)
y_prob_sample = best_rf.predict_proba(X_sample_scaled)[:, 1]

result_df = sample_data.copy()
result_df["Actual"] = y_actual.values
result_df["Predicted"] = y_pred_sample
result_df["Correct"] = result_df["Actual"] == result_df["Predicted"]
result_df["Probability"] = y_prob_sample

label_map = {0: "Not Cancelled", 1: "Cancelled"}
result_df["Actual Label"] = result_df["Actual"].map(label_map)
result_df["Predicted Label"] = result_df["Predicted"].map(label_map)

display_columns = ["Actual", "Predicted", "Actual Label", "Predicted Label", "Probability", "Correct"]
print(result_df[display_columns])

correct = result_df["Correct"].sum()
print("\n" + "="*60)
print("Correct Predictions :", correct)
print("Wrong Predictions   :", 10 - correct)
print("Sample Accuracy     :", round((correct/10)*100, 2), "%")
print("="*60)


In [ ]:
print("Best Hyperparameters:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

print("\nFinal Tuned Random Forest Test Accuracy :", round(test_acc, 4))
print("Final Tuned Random Forest ROC AUC        :", round(roc, 4))
